<h2> <strong> Goal Oriented Agent</strong></h2>
<p>
Let’s create a Goal-Oriented Agent for the doctor appointment scenario.
</p>

<p>
Unlike the task-oriented agent, this one will:
<br>
<ul>
<li>Take a final goal (“Book a dermatology appointment tomorrow afternoon”)</li>

<li>Decide all the necessary steps automatically:</li>

<li>Find doctors</li>

<li>Check availability</li>

<li>Book the appointment </li>

<li>Return a final confirmation in one go </li></ul>
</p>

We’ll still use dynamic tools like before, but the agent now plans multiple steps toward a goal.

In [ ]:
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain.tools import tool

In [ ]:
import utils

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chat_models import ChatOpenAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools import tool

**Let’s create a Goal-Oriented Agent for the doctor appointment scenario.**


<li>Take a final goal (“Book a dermatology appointment tomorrow afternoon”)</li>li>

<li> Decide all the necessary steps automatically: </li>

<li> Find doctors </li>

<li> Check availability </li>

<li> Book the appointment & Return a final confirmation in one go </li>
<p></p>

*We’ll still use dynamic tools like before, but the agent now plans multiple steps toward a goal.*

<strong> In production scenario: In a real-world healthcare system, this agent would interact with live systems, and several additional considerations apply: </strong>

| Component             | Currently                  | Production Version                                                                  |
| --------------------- | --------------------------- | ----------------------------------------------------------------------------------- |
| Doctor Database       | Hardcoded dictionary        | Connects to live hospital/clinic database (EMR/EHR system)                          |
| Appointment Slots     | Static slots in code        | Pulled in **real-time** from scheduling system/API                                  |
| Booking Tool          | Returns static confirmation | Calls hospital API to **actually book** appointment and handle conflicts            |
| Error Handling        | None                        | Handle scheduling conflicts, unavailable slots, API failures, retries               |
| Security & Compliance | None                        | Must enforce **HIPAA/GDPR compliance**, encrypt sensitive data, user authentication |
| Logging & Auditing    | Verbose prints              | Centralized logging for monitoring, auditing, and compliance                        |
| Autonomy              | Controlled in code          | Could integrate **human oversight** or approval for critical tasks                  |
| Scalability           | Single-user simulation      | Multi-user, multi-agent orchestration for enterprise systems                        |
<p>
Contrasted with task-oriented agents, goal-oriented agents can complete workflows end-to-end, improving efficiency in enterprise use cases like healthcare scheduling, logistics, and personal assistants.
</p>

In [ ]:
# ----------------------
# Simulated Data (same as task-oriented)
# ----------------------
DOCTORS_DB = {
    "Dermatology": ["Dr. Smith", "Dr. Kim"],
    "Cardiology": ["Dr. Patel", "Dr. Lee"],
    "Neurology": ["Dr. Jones", "Dr. Wang"]
}

APPOINTMENT_SLOTS = {
    "Dr. Smith": ["Tomorrow 10AM", "Tomorrow 3PM"],
    "Dr. Kim": ["Tomorrow 11AM", "Tomorrow 4PM"],
    "Dr. Patel": ["Jan 21 9AM", "Jan 22 2PM"],
    "Dr. Lee": ["Jan 20 10AM", "Jan 22 1PM"],
    "Dr. Jones": ["Jan 23 11AM", "Jan 24 3PM"],
    "Dr. Wang": ["Jan 21 9AM", "Jan 24 2PM"]
}

In [ ]:
# Tools

@tool
def FindDoctorsTool(specialty: str):
    """Return available doctors for the given specialty."""
    doctors = DOCTORS_DB.get(specialty, [])
    if doctors:
        return f"Available {specialty} doctors: {', '.join(doctors)}"
    return f"No doctors found for specialty: {specialty}"

@tool
def CheckAvailabilityTool(doctor_name: str):
    """Return available appointment slots for the selected doctor."""
    slots = APPOINTMENT_SLOTS.get(doctor_name, [])
    if slots:
        return f"Available slots for Dr. {doctor_name}: {', '.join(slots)}"
    return f"No available slots found for Dr. {doctor_name}"

@tool
def BookAppointmentTool(details: str):
    """Book an appointment and return confirmation."""
    return f"✅ Appointment successfully booked for: {details}"

tools = [FindDoctorsTool, CheckAvailabilityTool, BookAppointmentTool]

In [ ]:
GOAL_SYSTEM_PROMPT = """You are a goal-oriented healthcare assistant.

Your job is to plan and execute all the steps required to achieve the user's goal:
1. Find appropriate doctors for the requested specialty.
2. Check availability slots for that doctor.
3. Book the appointment.

Once all steps are completed successfully, provide a Final Answer containing:
- Doctor chosen
- Appointment slot
- Confirmation message"""

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system", GOAL_SYSTEM_PROMPT
    ),
    (
        "human", "User Goal: {input}"
    ),
    MessagesPlaceholder(
        variable_name="agent_scratchpad"
    )
])

In [ ]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True       # Show reasoning steps
    #,max_iterations=10   # Allow multiple steps to achieve goal
)

In [ ]:
# Run Goal-Oriented Task

response = executor.invoke({"input": "Book a Neurology appointment for 3PM afternoon"})
print("\n--- Goal-Oriented Agent Output ---")
print(response["output"])